# Maternal Instinct · Model A · Across Generations

คำถามหลัก: PPO ที่มี Reward จากสุขภาวะลูก ร่วมกับการคัดเลือกข้ามรุ่น ทำให้แม่รุ่นใหม่ดูแลได้ก่อนเรียนรู้เลี้ยงลูกเองมากขึ้นหรือไม่?

ใช้ Model A เท่านั้น แม่ดูแลตัวเองและลูก ไม่มีงาน ลูกค่อย ๆ ช่วยตัวเองตามวัย 0–7 / 7–14 / 14–20 วัน เดินทุก 5 / 3 / 1 Steps ตามวัย ช่วงกลางหยิบอาหารต้องมีแม่ใกล้และ OFFER ลูกที่รอดและได้รับเลือกจึงเป็นแม่รุ่นถัดไป โดยเริ่มจากพันธุกรรมพร้อม mutation ไม่รับน้ำหนักหลัง PPO ความจำหรือ optimizer ของวัยเด็ก

Zero-shot A0 ทดสอบสำเนาน้ำหนักตั้งต้นของแม่แต่ละรุ่นในบ้านใหม่ 20 วัน ไม่ฝึกเพิ่ม ไม่มี Reward ดูแลลูก และไม่กลับเข้าสายฝึก รุ่น 0 คือ baseline ของผู้ก่อตั้ง ผลใหม่อยู่ outputs/2_model ต้องเริ่ม Run ใหม่ ไม่ Resume checkpoint กติกาเก่า


## ลำดับใช้งาน

1. ใช้ Python 3.12 บนเครื่องใหม่ รัน Cell **0. ติดตั้งแพ็กเกจ** โดยเปลี่ยน INSTALL_PACKAGES เป็น True แล้วรัน Cell นี้เอง จากนั้นเปลี่ยนกลับ False และ Restart Kernel
2. เลือก Kernel Python 3.12 (Maternal #3) แล้วรัน Setup และ Config
3. กด Check balance เมื่อต้องการตรวจสมดุล ก่อนเริ่มเทรนผ่านปุ่มของหัวข้อ 2 หรือ 3
4. การเปิด Notebook/สร้างหน้าควบคุมไม่เริ่ม Simulation หรือเทรน Cell ติดตั้งมีค่าเริ่มต้น False จึงไม่ติดตั้งเองเมื่อ Run All
5. หลังฝึกใช้ checkpoint ดูโมเดล ประเมินเพิ่มเติม หรือสร้างรายงาน

เปิด Notebook จากโฟลเดอร์โปรเจกต์ #3 โดยตรง บนเครื่องนี้ใช้ C:\Code\Instinct for AI\Maternal Instinct For AI #3 เป็นพาธหลัก ไม่ค้นหาหรือคัดลอกโปรเจกต์ข้างเคียง หากย้าย clone ไปที่อื่นให้เปิดจากโฟลเดอร์นั้นหรือกำหนด PROJECT_DIR_OVERRIDE

หยุดงานและรอจนบันทึกเสร็จก่อนติดตั้งแพ็กเกจ Restart Kernel หรือเปลี่ยนโค้ด


## 0. ติดตั้งแพ็กเกจบนเครื่องใหม่ (รันเองครั้งแรก)

ใช้ Kernel Python 3.12 ตั้ง `INSTALL_PACKAGES = True` แล้วรัน Cell ถัดไป จะติดตั้ง requirements.txt และลงทะเบียน Kernel ใน Python ที่กำลังใช้อยู่ เตรียม PyTorch แบบ CUDA เมื่อพบ NVIDIA/driver ที่รองรับ มิฉะนั้น CPU มีการตรวจ CUDA ใน process แยกหลังติดตั้งและใช้ CPU หากตรวจไม่ผ่าน

เลือก `TORCH_BUILD` เองได้: auto / cpu / cu126 / cu128 / cu130 โดยค่าเริ่มต้นเป็น auto ชุดติดตั้งอ้างอิง [PyTorch 2.9.1](https://pytorch.org/get-started/previous-versions/#v291) ติดตั้งเฉพาะแพ็กเกจ Python ไม่ติดตั้ง Python หรือ NVIDIA driver ให้

หลังจบเปลี่ยน INSTALL_PACKAGES กลับ False แล้ว Restart Kernel เลือก Python 3.12 (Maternal #3) และรัน Setup/Config ไม่เริ่มเทรนหรือประเมินจาก Cell นี้


In [ ]:
from pathlib import Path
import sys
import os

# Optional override for a relocated clone. Only project #3 is accepted.
PROJECT_DIR_OVERRIDE = globals().get("PROJECT_DIR_OVERRIDE", None)
PRIMARY_PROJECT = Path(r"C:\Code\Instinct for AI\Maternal Instinct For AI #3")

def is_project_three(folder):
    markers = ("Maternal_Instinct_Experiment.ipynb", "requirements.txt", "maternal/config.py", "maternal/notebook.py")
    if not all((folder/name).is_file() for name in markers):
        return False
    return 'PROJECT_ID = "maternal-instinct-3"' in (folder/"maternal/config.py").read_text(encoding="utf-8-sig")

if PROJECT_DIR_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_DIR_OVERRIDE).expanduser().resolve()
    if not is_project_three(PROJECT_ROOT):
        raise FileNotFoundError("PROJECT_DIR_OVERRIDE must point to project #3 itself, not the parent or another project.")
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    if os.name == "nt":
        candidates.append(PRIMARY_PROJECT)
    PROJECT_ROOT = next((folder.resolve() for folder in candidates if is_project_three(folder)), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open this notebook from the project #3 folder, or set PROJECT_DIR_OVERRIDE to its cloned location.")

existing = sys.modules.get("maternal")
if existing is not None and not Path(existing.__file__).resolve().is_relative_to(PROJECT_ROOT):
    raise RuntimeError("Restart Kernel: a different project is already imported.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# Change to True and run THIS CELL when installing on a new machine.
# False makes Run All skip package installation.
INSTALL_PACKAGES = False
TORCH_BUILD = "auto"  # auto / cpu / cu126 / cu128 / cu130

if not INSTALL_PACKAGES:
    print("Installation skipped. Set INSTALL_PACKAGES = True and run this cell when needed.")
    print("Project:", PROJECT_ROOT)
else:
    import re
    import shutil
    import subprocess

    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("Select a Python 3.12 kernel first. This cell installs Python packages, not Python itself.")
    for name in ("preview_panel", "balance_panel", "training_panel", "batch_panel", "resume_panel", "model_panel", "evaluation_panel"):
        if getattr(globals().get(name), "active", False):
            raise RuntimeError("Stop the active job and save before installing packages.")
    if "torch" in sys.modules or "maternal" in sys.modules:
        raise RuntimeError("Restart Kernel, then run only this installer cell before Setup/Config.")
    if TORCH_BUILD not in ("auto", "cpu", "cu126", "cu128", "cu130"):
        raise ValueError("TORCH_BUILD must be auto, cpu, cu126, cu128 or cu130")

    chosen = TORCH_BUILD
    if chosen == "auto":
        chosen = "cpu"
        smi = shutil.which("nvidia-smi")
        if smi and sys.platform in ("win32", "linux"):
            try:
                output = subprocess.check_output([smi], stderr=subprocess.STDOUT, text=True, timeout=15)
                match = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", output)
                if match:
                    driver_cuda = tuple(map(int, match.groups()))
                    # Official wheels for the pinned PyTorch version. The small
                    # post-install CUDA probe below verifies actual GPU support.
                    for version, build in (((12, 8), "cu128"), ((12, 6), "cu126")):
                        if driver_cuda >= version:
                            chosen = build
                            break
            except (OSError, subprocess.SubprocessError) as error:
                print("NVIDIA detection unavailable; using CPU:", error)
    if sys.platform == "darwin" and chosen != "cpu":
        raise ValueError("CUDA wheels are for Windows/Linux. Use auto or cpu on macOS.")

    requirements = PROJECT_ROOT/"requirements.txt"
    torch_spec = next(line.strip() for line in requirements.read_text(encoding="utf-8").splitlines()
                      if line.strip().startswith("torch=="))
    python = sys.executable
    index = None if sys.platform == "darwin" else "https://download.pytorch.org/whl/" + chosen
    print("Project:", PROJECT_ROOT)
    print("Installing into kernel Python:", python)
    print("PyTorch build:", chosen)
    subprocess.check_call([python, "-m", "pip", "install", "--upgrade", "pip"])
    command = [python, "-m", "pip", "install", "--force-reinstall", torch_spec]
    if index:
        command += ["--index-url", index]
    subprocess.check_call(command)
    subprocess.check_call([python, "-m", "pip", "install", "-r", str(requirements)])

    if chosen != "cpu":
        # Separate process: no imported torch/DLL remains in the notebook kernel.
        probe = ("import torch; "
                 "assert torch.cuda.is_available(), 'CUDA unavailable'; "
                 "print('CUDA verified:', torch.cuda.get_device_name(0), torch.ones(4, device='cuda').sum().item())")
        try:
            subprocess.check_call([python, "-c", probe])
        except subprocess.CalledProcessError:
            print("The CUDA wheel cannot run on this GPU/driver. Installing the CPU build instead.")
            subprocess.check_call([python, "-m", "pip", "install", "--force-reinstall", torch_spec,
                                   "--index-url", "https://download.pytorch.org/whl/cpu"])
            subprocess.check_call([python, "-m", "pip", "install", "-r", str(requirements)])
    subprocess.check_call([python, "-m", "pip", "check"])
    subprocess.check_call([python, "-m", "ipykernel", "install", "--user", "--name", "maternal3",
                           "--display-name", "Python 3.12 (Maternal #3)"])
    print("Installed. Set INSTALL_PACKAGES back to False, Restart Kernel, select Python 3.12 (Maternal #3), then run Setup and Config.")
    print("No model, simulation, training or evaluation was started.")


## Setup

โหลดเฉพาะโปรเจกต์ #3 และหน้าควบคุม ยังไม่เริ่มงาน


In [1]:
from pathlib import Path
import sys
import os

# Optional override for a relocated clone. Only project #3 is accepted.
PROJECT_DIR_OVERRIDE = globals().get("PROJECT_DIR_OVERRIDE", None)
PRIMARY_PROJECT = Path(r"C:\Code\Instinct for AI\Maternal Instinct For AI #3")

def is_project_three(folder):
    markers = ("Maternal_Instinct_Experiment.ipynb", "requirements.txt", "maternal/config.py", "maternal/notebook.py")
    if not all((folder/name).is_file() for name in markers):
        return False
    return 'PROJECT_ID = "maternal-instinct-3"' in (folder/"maternal/config.py").read_text(encoding="utf-8-sig")

if PROJECT_DIR_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_DIR_OVERRIDE).expanduser().resolve()
    if not is_project_three(PROJECT_ROOT):
        raise FileNotFoundError("PROJECT_DIR_OVERRIDE must point to project #3 itself, not the parent or another project.")
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    if os.name == "nt":
        candidates.append(PRIMARY_PROJECT)
    PROJECT_ROOT = next((folder.resolve() for folder in candidates if is_project_three(folder)), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open this notebook from the project #3 folder, or set PROJECT_DIR_OVERRIDE to its cloned location.")

existing = sys.modules.get("maternal")
if existing is not None and not Path(existing.__file__).resolve().is_relative_to(PROJECT_ROOT):
    raise RuntimeError("Restart Kernel: a different project is already imported.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import importlib.util
for panel_name in ("preview_panel", "balance_panel", "training_panel", "batch_panel", "resume_panel", "model_panel", "evaluation_panel"):
    if getattr(globals().get(panel_name), "active", False):
        raise RuntimeError("Stop the active job and wait for saving before running Setup again.")
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the project Python 3.12 kernel.")
if importlib.util.find_spec("ipywidgets") is None:
    raise ImportError("Run the optional installer cell first, then Restart Kernel. Nothing was installed automatically.")
loaded_config = sys.modules.get("maternal.config")
if loaded_config is not None and getattr(loaded_config, "PROTOCOL", "") != "maternal-house-v3.1-effective-care-reward":
    raise RuntimeError("Restart Kernel to load the current environment rules.")

from IPython.display import display
from maternal.config import ExperimentConfig
from maternal.notebook import PreviewPanel, BalancePanel, TrainingPanel, ResumePanel, FrozenModelPanel, EvaluationPanel, ReportsPanel
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Setup complete. No simulation, models or training have started.")


Project: C:\Code\Instinct for AI\Maternal Instinct For AI #3
Python: c:\Code\Instinct for AI\Maternal Instinct For AI #3\.venv\Scripts\python.exe
Setup complete. No simulation, models or training have started.


## เทคนิคที่ใช้: Model A — PPO + Baldwinian Evolution

การสืบทอดและ Fitness เดิมไม่เปลี่ยน: ส่งเฉพาะ genome ตั้งต้น + mutation ไม่ส่ง policy หลังเรียน ความจำ หรือ optimizer เมื่อโตเป็นแม่ A0 ใช้สำเนาจาก genome ตรึงน้ำหนัก ไม่มี PPO และไม่มี Reward ดูแลลูก

Reward ดูแลลูกใหม่ให้แม่เฉพาะระหว่าง Train และเมื่อแม่ยังมีชีวิต:

| ส่วน | สูตร/เงื่อนไข |
|---|---|
| แก้ความหิว | 0.6 × Food ที่เพิ่มจริงในช่วงต่ำกว่า 50 / 30 เฉพาะการป้อนของแม่ หรืออาหารที่แม่ให้แล้วลูกกินจริง |
| ลดความกลัว | 0.2 × ความกลัวส่วนที่เกิน 60 ที่แม่ปลอบลดลงจริง / 12 ต้องปลอบสำเร็จตามกติกาการยินยอม |
| รักษาสุขภาวะ | 0.001 × welfare ต่อ Step จากเดิม 0.02 ไม่เพิ่มคะแนนแค่เลือก Action |
| ความต้องการค้าง | −0.005 × [max(0,30−Food)/30 + max(0,Fatigue−70)/30 + max(0,Fear−60)/40] เมื่อลูกยังมีชีวิต |
| เสีย Health | −0.02 × Health ที่ลูกเสียใน Step นั้น |
| ลูกตาย | −2 ครั้งเดียวตอนตาย |
| จบ Episode | +2 × Health ลูก/100 เฉพาะเมื่อแม่และลูกรอดครบ Episode ไม่ซ้ำระหว่างทาง |

welfare = (Health/100) × min(Food/40,1) × (1−Fatigue/100) × (1−Fear/100); ลูกตายเป็นศูนย์ อาหารที่ลูกหาเองไม่รับโบนัสว่าแม่ป้อน แต่สุขภาวะดีได้ maintenance เล็กน้อย การส่งอาหารไว้ไม่ให้โบนัสล่วงหน้า เพื่อไม่จูงใจให้ส่งซ้ำโดยลูกไม่กิน

Reward ดูแลตัวเองเดิม ความหิว/ความเสียหายจริง Fitness และ Zero-shot คงเดิม บรรพบุรุษได้รับการสอนผ่าน Reward จึงทดสอบการสืบทอดแนวโน้มก่อนประสบการณ์ของรุ่นใหม่ ไม่ใช่พิสูจน์ว่าไม่เคยมีแรงจูงใจในสายวิวัฒนาการ


## Config

pilot คือ 16 ครอบครัว × 20 Generations ต่อ arm/seed แต่ละ Episode 20 วัน × 200 Steps Zero-shot ใช้เวลาเท่ากัน โหมด smoke ย่อชีวิตเพื่อตรวจระบบเท่านั้น

ก่อน 7 วันแม่ป้อนโดยตรง; 7–14 วันกินเองและหยิบอาหารเมื่อแม่อยู่ระยะ 1 ช่องและส่งสัญญาณ OFFER (เว้น 40 Steps); ตั้งแต่ 14 วันหยิบเองได้โดยไม่ต้องมีแม่อยู่ใกล้และเว้นระยะลดจาก 20 เป็น 4 Steps เมื่ออายุครบ 20 วัน ลูกยังไม่เปลี่ยน role จนกว่าจะได้รับเลือกขึ้นรุ่นถัดไป

Fitness = ลูกรอด × (0.5 + 0.3 × Health เฉลี่ย/100 + 0.2 × คะแนนความพร้อมเป็นผู้ใหญ่) คะแนนความพร้อมทดสอบสำเนาหลังจบวัยลูก โดยจำลอง role เป็นแม่ ไม่เปลี่ยนความสามารถของลูกจริงใน Episode

ผลใหม่อยู่ outputs/2_model/ ผลเก่าคงเดิม ห้ามรวมกติกาต่างกัน Device เริ่มต้น auto เลือก CUDA เมื่อ PyTorch ใช้ได้ มิฉะนั้น CPU แสดงอุปกรณ์ตอนกดเริ่มงาน


In [2]:
from pathlib import Path
import sys
import os

# Optional override for a relocated clone. Only project #3 is accepted.
PROJECT_DIR_OVERRIDE = globals().get("PROJECT_DIR_OVERRIDE", None)
PRIMARY_PROJECT = Path(r"C:\Code\Instinct for AI\Maternal Instinct For AI #3")

def is_project_three(folder):
    markers = ("Maternal_Instinct_Experiment.ipynb", "requirements.txt", "maternal/config.py", "maternal/notebook.py")
    if not all((folder/name).is_file() for name in markers):
        return False
    return 'PROJECT_ID = "maternal-instinct-3"' in (folder/"maternal/config.py").read_text(encoding="utf-8-sig")

if PROJECT_DIR_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_DIR_OVERRIDE).expanduser().resolve()
    if not is_project_three(PROJECT_ROOT):
        raise FileNotFoundError("PROJECT_DIR_OVERRIDE must point to project #3 itself, not the parent or another project.")
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    if os.name == "nt":
        candidates.append(PRIMARY_PROJECT)
    PROJECT_ROOT = next((folder.resolve() for folder in candidates if is_project_three(folder)), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open this notebook from the project #3 folder, or set PROJECT_DIR_OVERRIDE to its cloned location.")

existing = sys.modules.get("maternal")
if existing is not None and not Path(existing.__file__).resolve().is_relative_to(PROJECT_ROOT):
    raise RuntimeError("Restart Kernel: a different project is already imported.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from maternal.config import ExperimentConfig, PROTOCOL
if PROTOCOL != "maternal-house-v3.1-effective-care-reward":
    raise RuntimeError("Restart Kernel to load the updated home environment.")

CONFIG = ExperimentConfig.profile(mode="pilot", arm="A", seed=0)

# แก้ค่าได้ก่อนสร้างหน้าควบคุม เช่น:
# CONFIG.population = 8
# CONFIG.generations = 10
# CONFIG.eval_candidates = 2
# อย่าย่ออายุในงานหลักโดยไม่เปลี่ยน protocol และรายงานเหตุผล

CONFIG.validate()
planned_care_steps = CONFIG.population * CONFIG.generations * CONFIG.world.horizon
print(f"Profile: {CONFIG.mode} | Arm: {CONFIG.arm} | Seed: {CONFIG.seed}")
print(f"{CONFIG.population} families × {CONFIG.generations} generations")
print(f"{CONFIG.world.days} days × {CONFIG.world.steps_per_day} steps per episode")
print(f"Care budget per run: {planned_care_steps:,} joint steps, excluding fitness and frozen evaluation")
print("Parameters prepared only. No model has been created.")
print("Home: mother self-care and childcare / No work action or work reward")


Profile: pilot | Arm: A | Seed: 0
16 families × 20 generations
20 days × 200 steps per episode
Care budget per run: 1,280,000 joint steps, excluding fitness and frozen evaluation
Parameters prepared only. No model has been created.
Home: mother self-care and childcare / No work action or work reward


## 1. พรีวิวโลก

กด **Start preview** เพื่อดูโลกและสถานะ เป็น scripted controller สำหรับตรวจการแสดงผล **ไม่ใช่ Agent ที่เทรนแล้ว** ไม่มี PPO หรือการบันทึกโมเดลจากพรีวิว

In [ ]:
preview_panel = PreviewPanel(CONFIG)
display(preview_panel.widget)

## 1.1 ตรวจสมดุลก่อนเทรน

กด **Check balance** เมื่อต้องการตรวจด้วยกฎสี่แบบ: ไม่ช่วย / ช่วยน้อยเป็นช่วง ๆ / ดูแลตามความต้องการ / ช่วยเกินจำเป็น ในชีวิตปกติ สถานการณ์ข้อสอบ และ mother_loss ที่จำลองแม่ตายวันที่ 5 ไม่โหลดโมเดลและไม่ฝึก กด Stop check ได้

ผลอยู่ใน `outputs/2_model/balance_check_.../summary.csv` และ `episodes.csv` อ่าน Health ลูก การรอด เวลาที่ความต้องการค้าง และการแทรกแซง เทียบทั้งสี่แบบก่อนลงทุนเทรนหลายรุ่น ผลต้องตรวจจริง ไม่ถือว่ากติกาสมดุลแล้วจากการแก้ตัวเลขเพียงอย่างเดียว โหมด smoke สั้นกว่าวัยพึ่งแม่ จึงไม่ใช้ยืนยันสมดุลตลอดชีวิต

การรันเซลล์นี้สร้างหน้าควบคุมเท่านั้น ยังไม่เดิน Simulation จนกดปุ่ม


mother_loss ใช้ตรวจกลไกการพึ่งแม่แยกต่างหาก ไม่เอาการตายที่กำหนดไว้ในข้อสอบนี้ไปรวมเป็นคะแนนหลักของโมเดล ดู child_orphan_steps และ orphaned_child_alive_at_end ด้วย หากแม่ตายใกล้หมด Episode ลูกอาจยังมี Food/Health เหลือและยังไม่ตายภายในเวลาที่เหลือ

light_help ช่วยเฉพาะช่วง 2/4 วันแรก (ครึ่งวัน) ทุก 4 วัน เวลาที่เหลือดูแลตัวเอง เป็นกฎอ้างอิง ไม่ใช่โมเดลเพิ่ม การรอดขึ้นกับความต้องการที่ได้รับการตอบสนองจริง ไม่มีการสุ่มตายหรือบังคับรอดตามจำนวนครั้งที่ช่วย ต้องกด Check balance เองเพื่อดูว่าดูแลเพียงพอหรือไม่


In [ ]:
balance_panel = BalancePanel(CONFIG)
display(balance_panel.widget)


## 2. เทรน Model A หนึ่ง Seed

เลือก Seed แล้วกด Start training Cell นี้สร้างหน้าควบคุมเท่านั้น Pause พัก; Stop & save บันทึกก่อนหยุด View Off ปิดภาพเพื่อลด overhead

แต่ละรุ่นฝึก 20 วัน แล้วประเมินสำเนา A0 จากน้ำหนักก่อนเรียนในบ้านใหม่ โดยตรึงน้ำหนัก ไม่มี PPO และไม่มี Reward ลูก ผลทดสอบไม่ใช้คัดเลือก ระบุ Generation ตามแม่ที่ถูกทดสอบ: 0 = ผู้ก่อตั้ง, 1 = ลูกที่โตเป็นแม่ หากไม่มีลูกรอดจะประเมินรุ่นปัจจุบันแล้วหยุด extinct


In [ ]:
training_panel = TrainingPanel(CONFIG)
display(training_panel.widget)

## 3. เทรน Model A หลาย Seed ต่อกัน

ใช้แทนข้อ 2 เลือก Seeds ที่ต้องการ เช่น 0, 1, 2 จะรัน A ตามลำดับรวม 3 Runs ไม่ใช่ 12 Runs แต่ละ Run มีผลและ checkpoint แยกกัน ไม่เริ่มจนกด Start batch และไม่สุ่มเริ่มใหม่เมื่อสูญพันธุ์


In [3]:
batch_panel = TrainingPanel(CONFIG, arms=("A",), seeds=(1,2,3))
display(batch_panel.widget)

## 4. รันต่อ

กรอก path เช่น `outputs/run_.../models/latest.pt` แล้วกด **Resume training** ใช้ checkpoint ของ #3 ที่สร้างเองและไว้ใจได้เท่านั้น โค้ดตรวจ protocol และ source hashes ก่อนรันต่อ

หากหยุดกลาง probes ระบบอาจคำนวณ probe ที่ค้างใหม่แบบ frozen ไม่เปลี่ยนน้ำหนัก train และไม่นับแถวผลซ้ำ

In [ ]:
resume_panel = ResumePanel(CONFIG)
display(resume_panel.widget)

## 5. ดูพฤติกรรม Zero-shot A0

เลือก checkpoint ที่จบรุ่นแล้ว Family และสถานการณ์ กด Show model เพื่อดูสำเนาน้ำหนักที่สืบทอด ล้างความจำ ไม่เปิด PPO หรือ Reward ดูแลลูก ลูกในข้อสอบใช้กฎเดียวกันเพื่อเทียบแต่ละรุ่น ทดสอบครบ 20 วัน ไม่ใช่การเล่นเหตุการณ์ train ย้อนหลัง


In [ ]:
model_panel = FrozenModelPanel(CONFIG)
display(model_panel.widget)

## 6. ประเมิน Zero-shot เพิ่มเมื่อกดเอง

เลือก checkpoint ที่จบรุ่นแล้วและเงื่อนไขข้อสอบ ทดสอบ A0 เท่านั้น 20 วัน × 200 Steps ด้วยสำเนาจากพันธุกรรม น้ำหนักตรึง ความจำใหม่ ไม่มี PPO และไม่มี Reward ดูแลลูก ผลแยก monitor/holdout และ matched/native ไม่แก้น้ำหนักหรือผลเดิม

กฎอ้างอิง no_help / light_help / appropriate_help / always_help ใช้ประกอบการอ่านผล ไม่ใช่โมเดลที่เทรนเพิ่ม ข้อสอบไม่มี competing_work


In [ ]:
evaluation_panel = EvaluationPanel(CONFIG)
display(evaluation_panel.widget)

## 7. กราฟและรายงาน: ผล Zero-shot เปลี่ยนตาม Generation หรือไม่?

Refresh runs เลือก Run → Build run report หรือเลือกหลาย Seed ของ Model A → Compare selected

กราฟหลัก a0_*_trajectory.png แสดงการรอดของลูก แม่ ทั้งคู่ และ Health ลูกเฉลี่ยตลอด Episode แถวบนคือผล A0 รายรุ่น แถวล่างคือผลต่างจาก Generation 0 ของ Seed เดียวกัน เส้นบางคือแต่ละ Seed เส้นหนาคือค่าเฉลี่ย Seed ที่มีผล

มี CSV change_from_gen0 และ run_counts บอกจำนวน Seed ที่มีผลแต่ละรุ่น หากสูญพันธุ์ไม่มีผลอนาคต ไม่เติมผลสำเร็จหรือเชื่อมเส้นต่อ ต้องรายงานจำนวน Run ที่สูญพันธุ์ด้วย ผล train เป็นข้อมูลประกอบ ไม่นำมารวมเป็นผล Zero-shot กราฟอื่นแสดงความต้องการค้าง การแทรกแซงเกินจำเป็น สุขภาพแยกวัย และกฎอ้างอิง

หลาย Seed ใช้ช่วงความไม่แน่นอนที่ระดับ Run; ครอบครัวหรือสถานการณ์ใน Run เดียวไม่ใช่การทดลองอิสระ ไม่มี B/C/D จึงยังแยกสาเหตุของ PPO และ selection ไม่ได้ และผลดีขึ้นไม่ใช่หลักฐานยืนยันสัญชาตญาณทางชีววิทยา

กราฟ training_care_reward.png แสดง Reward สะสมเฉลี่ยต่อครอบครัวแยก 7 ส่วน แกน X = Generation, Y = Reward ใช้วิเคราะห์การฝึกเท่านั้น ไม่ใช่หลักฐาน Zero-shot


In [ ]:
reports_panel = ReportsPanel()
display(reports_panel.widget)

## การอ่านผล

ดูความสำเร็จ zero-shot ควบคู่กับการรอด สุขภาวะ ความสามารถทำเอง และการแทรกแซงเกินจำเป็น จำนวนครั้งที่ป้อนมากไม่เท่ากับดูแลเหมาะสม และผลดีหลังเรียนยังไม่แสดงว่ามีพฤติกรรมก่อนประสบการณ์

สถานะ run อยู่ใน `manifest.json`: completed คือครบกำหนด, extinct คือไม่มีผู้รอดสร้างรุ่นถัดไป, stopped คือผู้ใช้หยุด, error คือเกิดข้อผิดพลาด

ยังไม่ควรสรุปสมมติฐานก่อนตรวจ collector, feasibility ของโลก และความไม่แน่นอนหลาย runs รายละเอียดข้อจำกัดและสูตรอยู่ใน [docs/PROTOCOL_TH.md](docs/PROTOCOL_TH.md) ส่วนวิธีติดตั้งและเริ่มใช้อยู่ใน [README.md](README.md)

แนวคิด Baldwinian: [Fernando et al. (2018)](https://arxiv.org/abs/1806.07917) การเรียนรู้ในชีวิตอาจช่วยวิวัฒนาการคัดเลือกความสามารถ แต่ไม่รับประกันว่าจะเกิดพฤติกรรมสำเร็จก่อนมีประสบการณ์